# Optimización

SGD, Adam y learning rate — cómo los modelos aprenden eficientemente

## Introducción

El descenso de gradiente actualiza cada peso restando learning_rate * gradient. Si weight=10, gradient=4, lr=0.1: new_weight = 10 - 0.1*4 = 9.6. Adam combina momentum y adaptive learning rates.

## Paso de Descenso de Gradiente

> La actualización es: new_weight = weight - learning_rate * gradient. Esto mueve el peso en dirección opuesta al gradiente, minimizando la loss.

In [ ]:
def gradient_descent_step(weight, gradient, learning_rate):
    return weight - learning_rate * gradient

print(gradient_descent_step(10.0, 4.0, 0.1))

## SGD (Stochastic Gradient Descent)

> SGD actualiza los pesos usando el gradiente de un batch pequeño en vez de todo el dataset. Es más rápido y puede escapar mínimos locales.

In [ ]:
print("=== SGD ===")

w = 10.0
lr = 0.1
gradient = 4.0

print(f"Peso inicial: {w}")
for i in range(5):
    w = w - lr * gradient
    print(f"  Step {i+1}: w = {w}")

print("\nTipos de SGD:")
tipos = {
    "Batch GD": "Todo el dataset por paso (lento, preciso)",
    "Mini-batch GD": "Batch de 32-256 muestras (balance)",
    "SGD": "1 muestra por paso (rápido, ruidoso)",
}
for nombre, desc in tipos.items():
    print(f"  {nombre}: {desc}")

## Momentum

> Momentum acumula gradientes pasados para acelerar convergencia. Ayuda a escapar mesetas y mínimos locales.

In [ ]:
print("=== Momentum ===")

def sgd_momentum(weights, gradients, lr, momentum=0.9):
    velocity = [0] * len(weights)
    for i in range(len(weights)):
        velocity[i] = momentum * velocity[i] - lr * gradients[i]
        weights[i] += velocity[i]
    return weights, velocity

w = [10.0, -5.0]
grads = [4.0, 2.0]

print(f"Pesos iniciales: {w}")
for epoch in range(3):
    w, v = sgd_momentum(w, grads, 0.1, momentum=0.9)
    print(f"  Epoch {epoch+1}: weights={w}")

## Adam (Adaptive Moment Estimation)

> Adam combina momentum y RMSProp. Usa momentos primer y segundo orden para adaptar el learning rate por parámetro.

In [ ]:
print("=== Adam ===")

import numpy as np

def adam_update(m, v, beta1=0.9, beta2=0.999, epsilon=1e-8):
    m_hat = m / (1 - beta1)
    v_hat = v / (1 - beta2)
    return m_hat, v_hat

print("Parámetros de Adam:")
print("  m: momento primer orden (media de gradientes)")
print("  v: momento segundo orden (media de gradientes²)")
print("  beta1=0.9, beta2=0.999")
print("  update: lr * m_hat / (sqrt(v_hat) + epsilon)")

print("\nPor qué Adam funciona:")
print("  1. Momentum ayuda en direcciones Consistentes")
print("  2. Learning rate adaptativo por parámetro")
print("  3. Bueno para problemas con gradientes dispersos")

## Learning Rate

> El learning rate controla qué tan grandes son los pasos. Muy alto = diverge. Muy bajo = converge lento.

In [ ]:
print("=== Learning Rate ===")

f = lambda x: x**2
df = lambda x: 2*x

lr_comparisons = {
    "Muy alto (0.5)": 0.5,
    "Alto (0.3)": 0.3,
    "Normal (0.1)": 0.1,
    "Bajo (0.01)": 0.01,
}

x = 10.0
for name, lr in lr_comparisons.items():
    x_cur = x
    for _ in range(10):
        x_cur = x_cur - lr * df(x_cur)
    print(f"  {name}: x_final = {x_cur:.4f}")

print("\nProblemas:")
print("  LR muy alto: oscila o diverge")
print("  LR muy bajo: converge muy lento")

## Learning Rate Scheduling

> Reducir el learning rate durante entrenamiento ayuda a converger mejor. Técnicas: step decay, exponential decay, cosine annealing.

In [ ]:
print("=== Learning Rate Scheduling ===")

lr_initial = 0.1
epochs = 10

print("Step Decay (reduce 50% cada 3 epochs):")
for epoch in range(epochs):
    lr = lr_initial * (0.5 ** (epoch // 3))
    print(f"  Epoch {epoch+1}: lr = {lr:.4f}")

print("\nExponential Decay:")
for epoch in range(epochs):
    lr = lr_initial * np.exp(-0.1 * epoch)
    if epoch < 3:
        print(f"  Epoch {epoch+1}: lr = {lr:.4f}")
print("  ...")

## Tips y Mejores Prácticas

> Adam es el optimizador default para la mayoría de casos. Solo usa SGD + momentum si tienes razón específica.

> Learning rate typical: 0.001 para Adam, 0.1 para SGD básico.

> Si la loss sube o oscila mucho, el LR es muy alto.

> Para fine-tuning, reduce el LR (ej: 1e-4 a 1e-5).

> Usa learning rate scheduling para converger mejor: reduce LR cuando la loss se estanca.

## Errores Comunes

### Learning rate muy alto

¿Por qué ocurre?
- Se usa LR default sin ajustar. Loss diverge o oscila.

Solución
- Reduce el LR. Si loss no baja en 2 epochs, es muy alto.

### No usar momentum

¿Por qué ocurre?
- SGD sin momentum converge muy lento en mesetas.

Solución
- Añade momentum=0.9. Ayuda a acelerar y escapar mínimos locals.

### Usar Adam siempre

¿Por qué ocurre?
- Adam funciona bien pero SGD converge mejor para algunos problemas.

Solución
- Para transfer learning, usa LR bajo con Adam. Para training from scratch, prueba SGD.

### LR scheduling muy agresivo

¿Por qué ocurre?
- Decaimiento muy rápido mata el aprendizaje antes de converger.

Solución
- Usa decay suave: cosine o warmup. No reduzcas más de 10x.